# CS336 Spring 2026 - Tokenizer Basic
---
> 目标是实现: Unicode -> UTF-8 -> Bytes -> 最小 Byte Tokenizer

此阶段先不考虑 BPE, regex, special tokens 等. 只考虑将底层表示彻底搞懂. 后面的 BPE 就是在此 bytes 上进行 merge.

## 1. 分清四个概念
- **Python str**
- **Unicode character**
- **Unicode code point**
- **bytes**(由 UTF-8 编码)

In [1]:
text = "你"

print("==========原始文本==========")
print(text)
print(len(text))
print(ord(text))  # ord() 将单个字符转换为对应的 Unicode 编码.

encoded = text.encode("utf-8")

print("==========编码后的二进制==========")
print(encoded)
print(len(encoded))
print(list(encoded))

==========原始文本==========
你
1
20320
==========编码后的二进制==========
b'\xe4\xbd\xa0'
3
[228, 189, 160]


这里的 `len("你")==1`, 但是 UTF-8 表示需要3个 bytes: `[228, 189, 160]`, 也就是
<center>
"你" -> [E4, BD, A0]
</center>

1. 这里的 `ord("你")` 得到的是 Unicode code point:
<center>
U+4F60
</center>
也就是十进制的 `20320`.

2. 这里的 `"你".encode("utf-8")` 得到的是:
<center>
E4 BD A0
</center>
也就是 UTF-8 编码.

## 2. `ord()` 不是 tokenizer 需要的 byte
> 这是第一个易出错的点. `ord()` 返回的是 Unicode code point 整数, 而不是 byte.

用上述的 `"你"` 作为例子, `ord("你")` 返回的是十进制 `20320`, 其 UTF-8 bytes 是 `228, 189, 160`. 这是两个层面的东西.
> 也可记忆为 byte 的十进制取值范围在 `0-255`, `ord()` 返回的数值超出了此范围, 因此不是 byte.

所以有:
$$
\text{Unicode code point}\neq \text{UTF-8 byte}
$$

## 3. Python 的 bytes
现在考虑 tokenizer 最重要的数据类型. 并且牢记两个 Python 的行为:
1. `bytes[index]` 返回 `int` 类型
2. `bytes[slice]` 返回 `bytes` 类型

In [2]:
x = b"abc"

print("=====x的数据类型=====")
print("x.type() =", type(x))  # <class 'bytes'>
print(list(x))  # [97, 98, 99]

print("=====x[0]=====")
print("x[0] =", x[0])  # 97
print("x[0].type() =", type(x[0]))  # <class 'int'>

print("=====x[0:1]======")
print("x[0:1] =", x[0:1])  # b'a'
print("x[0:1].type() =", type(x[0:1]))  # <class 'bytes'>

=====x的数据类型=====
x.type() = <class 'bytes'>
[97, 98, 99]
=====x[0]=====
x[0] = 97
x[0].type() = <class 'int'>
=====x[0:1]======
x[0:1] = b'a'
x[0:1].type() = <class 'bytes'>


## 4. 整数构造 byte
反过来, 可以从整数构造 byte, 即: `bytes([integer])`. 理论上, 对于任意 $0\leq i< 256$ 的整数, 都可以构造一个 byte. 因此天然获得一个大小为 256 的词表:

```Python
vocab = {i: bytes([i]) for i in range(256)}
```

后续的正式 BPE: 0-255, 这通常是基础的 byte token.

In [3]:
B = bytes([97])

print(B)
print("B.type()=", type(B))

b'a'
B.type()= <class 'bytes'>


## 5. 必须有全部256个 bytes.
假设 tokenizer 词表中只有 `a-z A-Z`, 那么遇到 中文, 日语, emoji等字符, 可能会出现 `<UNK>` 的问题. 但如果基础词表包含 256 个byte, 则 UTF-8 中任何文本最终都只是这些 bytes 的组合.

所以理论上
<center>
任何有效 UTF-8 str -> UTF-8 bytes -> 0-255 token IDs
</center>

这就是 byte-level tokenizer 一个非常漂亮的性质:
<center>
No OOV at byte level
</center>

## 6. UTF-8 round trip
UTF-8 round trip(UTF-8 往返)指的是: 将一段文本先编码为 UTF-8 字节序列, 再通过某种中间表示(如字节级 BPE 的映射)处理, 最后解码回原始文本, 并保证结果与原始文本完全一致, 无损.

In [4]:
text = "Hello 你好 🌍"

encoded = text.encode("utf-8")
decoded = encoded.decode("utf-8")

print(encoded)
print(decoded)

assert decoded == text


b'Hello \xe4\xbd\xa0\xe5\xa5\xbd \xf0\x9f\x8c\x8d'
Hello 你好 🌍


整个过程为:
$$
\text{Python str}\overset{\text{encode}}{\rightarrow} \text{bytes}\overset{\text{decode}}{\rightarrow} \text{Python str}
$$
并且对于原始合法 Python 字符串:

```Python
text.encode('utf-8').decode('utf-8') == text
```

In [5]:
text = "你"

data = text.encode("utf-8")

print("=====UTF-8编码后数据=====")
print(data)
print(data[0])
print(data[1])
print(data[2])

=====UTF-8编码后数据=====
b'\xe4\xbd\xa0'
228
189
160


In [6]:
print("=====尝试还原数据=====")
bytes(data[0]).decode("utf-8")

=====尝试还原数据=====


'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [7]:
print("=====正确还原数据=====")
bytes([228, 189, 160]).decode("utf-8")

=====正确还原数据=====


'你'

出现上述错误的原因是: E4 只是 "你" UTF-8 的一部分, 单独一个 byte 并不是一个完整的 UTF-8 character. 因此 BPE `decode()` 的重要原则为:
> 不要逐 token 做 UTF-8 decode, 而是先把所有 token 对应的 bytes 拼接起来, 最后整体 decode.

## 7. 构造最简单的 Tokenizer
先定义这一阶段的 tokenizer: ByteTokenizer. 它完全不做 compression. 其规则为

<center>
每一个 byte = 一个 token
</center>

例如:
```text
"abc" -> 97 98 99(UTF-8) -> token IDs=[97, 98, 99]

"你" -> 228 189 160(UTF-8) -> token IDs=[228, 189, 160]
```

In [8]:
# 实现一个最简单的 Tokenizer
class ByteTokenizer:
    def encode(self, text: str) -> list[int]:
        # 实现 str -> UTF-8 -> bytes -> list[int]
        text_bytes = text.encode("utf-8")

        return list(text_bytes)

    def decode(self, ids: list[int]) -> str:
        # 实现 list[int] -> bytes -> UTF-8 -> str
        text_bytes = bytes(ids)

        return text_bytes.decode("utf-8")

In [9]:
# 测试函数
def test_ascii():
    tokenizer = ByteTokenizer()

    text = "hello world"

    assert tokenizer.decode(tokenizer.encode(text)) == text


def test_chinese():
    tokenizer = ByteTokenizer()

    text = "你好，世界"

    assert tokenizer.decode(tokenizer.encode(text)) == text


def test_emoji():
    tokenizer = ByteTokenizer()

    text = "Hello 🌍🚀"

    assert tokenizer.decode(tokenizer.encode(text)) == text


def test_empty():
    tokenizer = ByteTokenizer()

    assert tokenizer.encode("") == []
    assert tokenizer.decode([]) == ""


def test_mixed_unicode():
    tokenizer = ByteTokenizer()

    text = "CS336: Hello 世界 🌍 café!"

    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)

    assert decoded == text

In [10]:
# 运行测试函数
test_ascii()
test_chinese()
test_emoji()
test_empty()
test_mixed_unicode()
print("All tests passed!")

All tests passed!


## 8. Compression Ratio
第一次计算压缩比(Compression Ratio), 其定义为:
$$
\text{Compression Ratio} = \frac{\text{UTF-8 bytes}}{\text{tokens}}
$$

对于 ByteTokenizer, 有:
$$
\text{number of bytes} = \text{number of tokens}
$$
所以无论什么, 理论上都有:
$$
\text{Compression Ratio} = 1
$$

这就是纯 Byte Tokenizer 的最大问题: 虽然万能, 但是完全没有压缩, 会导致后续操作的效率低下.

In [11]:
def compression_ratio(text: str, tokenizer: ByteTokenizer) -> float:
    # 计算 byte tokenizer 的压缩比
    num_bytes = len(text.encode("utf-8"))
    num_tokens = len(tokenizer.encode(text))

    return num_bytes / num_tokens

In [12]:
# 进行测试
samples = [
    "hello world",
    "你好世界",
    "こんにちは世界",
    "🌍🚀🔥",
]

for text in samples:
    encoded = text.encode("utf-8")

    print("text:", text)
    print("characters:", len(text))
    print("bytes:", len(encoded))
    print("byte values:", list(encoded))
    print()


text: hello world
characters: 11
bytes: 11
byte values: [104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100]

text: 你好世界
characters: 4
bytes: 12
byte values: [228, 189, 160, 229, 165, 189, 228, 184, 150, 231, 149, 140]

text: こんにちは世界
characters: 7
bytes: 21
byte values: [227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 228, 184, 150, 231, 149, 140]

text: 🌍🚀🔥
characters: 3
bytes: 12
byte values: [240, 159, 140, 141, 240, 159, 154, 128, 240, 159, 148, 165]



上述可以观察到:
1. 英文大部分时候有 $1 \text{character} \approx 1 \text{byte}$
2. 中文大部分时候有 $1 \text{characters} \approx 3 \text{bytes}$
3. emoji 经常有 $1 \text{character} \approx 4 \text{bytes}$

## 9. 正式的 BPE
最终的 BPE vocabulary 会类似
```Python
{
    97: b"a",
    98: b"b",
    # ...
    256: b"th",
    257: b"the",
    258: b"ing",
}
```
并且注意到 256, 257, 258 已经不再是 byte value. 因此正式的 BPE decoder 应该为:

<center>
token IDs -> voca[id] -> bytes fragment -> 全部拼接 -> UTF-8 decode
</center>


In [13]:
# 本节最后验证
tokenizer = ByteTokenizer()

assert tokenizer.encode("abc") == [97, 98, 99]
assert tokenizer.decode([97, 98, 99]) == "abc"

for text in ["", "hello", "你好", "🌍", "Hello 世界 🌍"]:
    assert tokenizer.decode(tokenizer.encode(text)) == text

print("All tests passed!")

All tests passed!
